# Notebook 03 - Statistische Analyse

- Korrelationsanalyse (Pearson r, p-Wert)
- Unabhaengiger t-Test (Welch)
- Lineare Regression
- Heatmap Korrelationsmatrix

In [ ]:
import pandas as pd, numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt, seaborn as sns
from itertools import combinations
pd.set_option('display.float_format', '{:.4f}'.format)
print('Imports OK')

In [ ]:
df = pd.read_csv('../data/inserate_bereinigt.csv')
print(f'{len(df)} Inserate, {df["stadt"].nunique()} Staedte')
df[['preis_chf','flaeche_m2','zimmer_anzahl','preis_pro_m2']].describe()

## 1. Deskriptive Statistik pro Stadt

In [ ]:
deskr = df.groupby('stadt')['preis_chf'].agg(n='count',mittelwert='mean',
    median='median',stdabw='std',minimum='min',maximum='max').round(0)
print('Preisstatistik pro Stadt (CHF/Monat):')
deskr.sort_values('mittelwert',ascending=False)

## 2. Korrelationsanalyse (Pearson r + p-Wert)

In [ ]:
num_cols = ['preis_chf','flaeche_m2','zimmer_anzahl','preis_pro_m2']
dn = df[num_cols].dropna()
print('Korrelationsanalyse (Pearson r + p-Wert):')
print('='*60)
print(f'{"V1":20} {"V2":20} {"r":>8} {"p-Wert":>12} Sig')
print('-'*60)
for c1,c2 in combinations(num_cols,2):
    r,p = stats.pearsonr(dn[c1],dn[c2])
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'n.s.'
    print(f'{c1:20} {c2:20} {r:8.4f} {p:12.6f} {sig}')
print('Signifikanz: *** p<0.001, ** p<0.01, * p<0.05, n.s.')

In [ ]:
fig,ax = plt.subplots(figsize=(8,6))
corr = dn.corr()
mask = np.triu(np.ones_like(corr,dtype=bool))
sns.heatmap(corr,mask=mask,annot=True,fmt='.3f',cmap='RdYlGn',
    vmin=-1,vmax=1,center=0,square=True,linewidths=0.5,ax=ax)
ax.set_title('Korrelationsmatrix - Pearson r')
plt.tight_layout()
plt.savefig('../data/korrelationsmatrix.png',dpi=150,bbox_inches='tight')
plt.show()
print('Gespeichert: korrelationsmatrix.png')

## 3. t-Test: Zuerich vs. andere Staedte
H0: Kein signifikanter Preisunterschied zu Zuerich (alpha=0.05)

In [ ]:
zh = df[df['stadt']=='Zuerich']['preis_chf'].dropna()
print(f'Zuerich: n={len(zh)}, Preis={zh.mean():.0f} CHF')
print('='*65)
print(f'{"Stadt":15}{"n":>5}{"Preis":>10}{"t":>12}{"p":>12} Entscheid')
print('-'*65)
for s in sorted([x for x in df['stadt'].unique() if x!='Zuerich' and pd.notna(x)]):
    g = df[df['stadt']==s]['preis_chf'].dropna()
    if len(g)<5: continue
    t,p = stats.ttest_ind(zh,g,equal_var=False)  # Welch's t-Test
    e = 'H0 ablehnen' if p<0.05 else 'H0 beibehalten'
    print(f'{s:15}{len(g):5}{g.mean():10.0f}{t:12.4f}{p:12.6f} {e}')
print('Methode: Welchs t-Test (ungleiche Varianzen)')

## 4. Lineare Regression: Preis ~ Flaeche

In [ ]:
x,y = dn['flaeche_m2'].values, dn['preis_chf'].values
m,b,r,p,se = stats.linregress(x,y)
print('Lineare Regression: Preis ~ Flaeche')
print(f'  Modell:  Preis = {m:.2f} x Flaeche + {b:.2f}')
print(f'  R2:      {r**2:.4f} ({r**2*100:.1f}% Varianzaufklaerung)')
print(f'  p-Wert:  {p:.2e}')
print(f'  -> Jeder m2 kostet CHF {m:.2f} mehr Miete')
fig,ax = plt.subplots(figsize=(9,6))
for s in df['stadt'].dropna().unique():
    sub = df[df['stadt']==s]
    ax.scatter(sub['flaeche_m2'],sub['preis_chf'],alpha=0.5,s=30,label=s)
xl = np.linspace(x.min(),x.max(),200)
ax.plot(xl,m*xl+b,'k-',lw=2.5,label=f'Regression (R2={r**2:.3f})')
ax.set_xlabel('Flaeche (m2)'); ax.set_ylabel('Preis (CHF)')
ax.set_title('Mietpreis vs. Flaeche')
ax.legend(fontsize=9); ax.grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig('../data/regression.png',dpi=150,bbox_inches='tight')
plt.show()